# 数据统计

In [1]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt
import math

from collections import defaultdict

In [2]:
import sys
sys.path.append('..')

In [3]:
# 项目方法
from songs.songs_libs import id_process

## 原始数据

In [4]:
file_path_prefix = "data/mayday/"

In [5]:
df_raw = pd.read_csv(file_path_prefix + 'cleared_song_data.csv')
df_raw['song_name'] = df_raw['song_name'].astype(str)
df_raw

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,0020I7sO0ayXhN,265,1224691200,突然好想你,36459,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,0006MmDz4Hl2Ud,273,1388332800,步步,451706,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,003PIMo40rxcAn,256,1124985600,知足,96397,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,249,1469030400,派对动物,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,002adz882rV5uh,209,1377792000,入阵曲,451706,2013-12-30
133,4932058,003PaRAX3j5wJk,生命有一种绝对,NaN,五月天,74,000Sp0Bz4JXH0o,时光机,0015r2I31enfaR,239,1038672000,生命有一种绝对,96353,2003-11-07
134,4830242,000PoJAV4NPMzW,温柔 (还你自由版),NaN,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,001ntd0y01uQ4g,426,1088611200,温柔 (还你自由版),96397,2005-08-26
135,4834459,002nqyCb1bUnk6,Enrich Your Life,NaN,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,0006oAnx03zXUC,166,1096560000,Enrich Your Life,96368,2004-11-01


In [17]:
df_words_raw = pd.read_csv(file_path_prefix + 'cleared_words_data.csv')
df_words_raw

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,来,v,17,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,107709592,能,v,6,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
2,107709592,人生,n,6,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
3,107709592,期待,v,5,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
4,107709592,有,v,4,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8674,519403016,天涯飞奔,id,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8675,519403016,回头,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8676,519403016,飞奔,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8677,519403016,请,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16


In [19]:
df_lyric_raw = pd.read_json(file_path_prefix + 'cleared_lyric_data.json')
df_lyric_raw

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text
0,107709592,后来的我们,2026-02-13 00:15:42,1,阿信,怪兽,,然后呢。他们说你的心似乎痊愈了。也开始有个人为你守护着。我该心安或是心痛呢。然后呢。其实我的...
1,447807,突然好想你,2026-02-13 00:13:39,1,阿信,阿信,,最怕空气突然安静。最怕朋友突然的关心。最怕回忆突然翻滚绞痛着不平息。最怕突然听到你的消息。想...
2,5131923,步步,2026-02-13 00:17:51,1,阿信,阿信/Cola,,空无一人的大街。闯入无人婚纱店。为你披上雪白誓言。世界已灰飞烟灭。而爱矗立高楼间。你是真的，...
3,4830286,知足,2026-02-13 00:21:58,1,阿信,阿信,,怎么去拥有一道彩虹。怎么去拥抱一夏天的风。天上的星星笑地上的人。总是不能懂不能觉得足够。如果...
4,106528423,派对动物,2026-02-13 00:00:48,1,阿信,阿信,,Let’s，go，party，party，all，night，oh，oh。Hey，lonel...
...,...,...,...,...,...,...,...,...
132,4996096,入阵曲,2026-02-13 00:27:52,1,阿信,怪兽,,当一座城墙，只为了阻挡。所有自由渴望。当一份信仰，再不能抵抗。遍地战乱饥荒。兰陵缭乱茫，天地...
133,4932058,生命有一种绝对,2026-02-13 00:01:25,1,阿信,阿信,,如果我，不曾走过这一遍。生命中，还有多少苦和甜美。那风中的歌声，孤单哽咽的声音是谁。回忆中，...
134,4830242,温柔 (还你自由版),2026-02-13 00:50:33,1,阿信,阿信,五月天,走在风中今天阳光。突然好温柔。天的温柔地的温柔。像你抱着我。然后发现你的改变。孤单的今后。如...
135,4834459,Enrich Your Life,2026-02-13 00:46:03,1,阿信,怪兽,,心情很晴朗。在大树下，笑着，和你乘凉。突然领悟了，幸福的形状。幸福不是多，而是遗忘。能遗忘生...


## 数据处理

In [20]:
# 拼接查询是否有歌词
df_songs = df_raw.merge(df_lyric_raw[['song_id', 'has_lyric']], on='song_id', how='left')
# id处理
df_songs = id_process(df_songs, is_words=False)
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,...,song_name_unique,album_id,publish_date,has_lyric,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,album_order
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,...,后来的我们,1393445,2016-07-21,1,song107709592,2016,自传,album1393445,自传,10
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,0020I7sO0ayXhN,265,...,突然好想你,36459,2008-10-23,1,song447807,2008,后青春期的诗,album36459,后青春期的诗,7
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,0006MmDz4Hl2Ud,273,...,步步,451706,2013-12-30,1,song5131923,2013,步步 自选作品辑,album451706,步步 自选作品辑,9
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,003PIMo40rxcAn,256,...,知足,96397,2005-08-26,1,song4830286,2005,知足 最真杰作选,album96397,知足 最真杰作选,5
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,249,...,派对动物,1393445,2016-07-21,1,song106528423,2016,自传,album1393445,自传,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,002adz882rV5uh,209,...,入阵曲,451706,2013-12-30,1,song4996096,2013,步步 自选作品辑,album451706,步步 自选作品辑,9
133,4932058,003PaRAX3j5wJk,生命有一种绝对,NaN,五月天,74,000Sp0Bz4JXH0o,时光机,0015r2I31enfaR,239,...,生命有一种绝对,96353,2003-11-07,1,song4932058,2003,时光机,album96353,时光机,3
134,4830242,000PoJAV4NPMzW,温柔 (还你自由版),NaN,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,001ntd0y01uQ4g,426,...,温柔 (还你自由版),96397,2005-08-26,1,song4830242,2005,知足 最真杰作选,album96397,知足 最真杰作选,5
135,4834459,002nqyCb1bUnk6,Enrich Your Life,NaN,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,0006oAnx03zXUC,166,...,Enrich Your Life,96368,2004-11-01,1,song4834459,2004,神的孩子都在跳舞,album96368,神的孩子都在跳舞,4


# 曲目数据

## 曲目词云

In [8]:
df_songs_for_cloud = df_songs[['song_name', 'album_fixed', 'album_order']].drop_duplicates().reset_index(drop=True).copy()
df_songs_for_cloud

,song_name,album_fixed,album_order
0,后来的我们,自传,10
1,突然好想你,后青春期的诗,7
2,步步,步步 自选作品辑,9
3,知足,知足 最真杰作选,5
4,派对动物,自传,10
...,...,...,...
132,入阵曲,步步 自选作品辑,9
133,生命有一种绝对,时光机,3
134,温柔 (还你自由版),知足 最真杰作选,5
135,Enrich Your Life,神的孩子都在跳舞,4


In [ ]:
dict_songs_for_cloud = df_songs_for_cloud.to_dict(orient='records')
with open(file_path_prefix + 'cloud_songs_data.json', 'w', encoding='utf-8') as f:
    json.dump(dict_songs_for_cloud, f, ensure_ascii=False, indent=4)

# 分词数据
名词： n, w
动词： v
形容词： v

In [10]:
def word_count_by_pos(df, pos, words_num=30, is_starts_with=True):
    if is_starts_with:
        word_cnt = df[df['pos'].str.startswith(
            pos, na=False)]['word'].value_counts().reset_index()
        word_sum = df[df['pos'].str.startswith(
            pos, na=False)].groupby('word')['freq'].sum().reset_index()
    else:
        word_cnt = df[df['pos']==pos]['word'].value_counts().reset_index()
        word_sum = df[df['pos']==pos].groupby('word')['freq'].sum().reset_index()
    if words_num:
        res = word_cnt.head(words_num).merge(word_sum, on='word', how='left')
    else:
        res = word_cnt.merge(word_sum, on='word', how='left')
    res['order'] = 100 - res.index
    res = res.rename(columns={
        'count': 'songs_num',
    })
    return res

In [ ]:
pos_list = ['n', 'v', 'a']
words_dict = defaultdict(dict)
for pos in pos_list:
    res_df = word_count_by_pos(df_words_raw, pos, 70)
    words_dict[pos] = res_df.to_dict('list')

with open(file_path_prefix + 'cloud_words_data.json', 'w', encoding='utf-8') as f:
    json.dump(words_dict, f, ensure_ascii=False, indent=4)

# 专辑数据

In [34]:
def get_album_stats(df_songs):
    # 1. 预处理：提取专辑基础信息并去重
    df_album = df_songs[['album_id', 'album_fixed', 'publish_date', 'album_order']].drop_duplicates()
    df_album['publish_date'] = df_album['publish_date'].astype(str)

    # 2. 聚合计算：一次性统计总数和纯音乐数
    # 使用 assign 创建一个临时列用于判断是否为纯音乐
    stats = df_songs.assign(
        is_instrumental = lambda x: (x['has_lyric'] == 0).astype(int)
    ).groupby('album_fixed').agg(
        songs_num=('album_fixed', 'count'),
        instrumental_num=('is_instrumental', 'sum')
    ).reset_index()

    # 3. 合并结果
    res =  df_album.merge(stats, on='album_fixed', how='left').fillna(0)
    return res.sort_values(by='album_order')

# 调用方法
df_album = get_album_stats(df_songs)
df_album

,album_id,album_fixed,publish_date,album_order,songs_num,instrumental_num
7,96215,第一张创作专辑,1999-07-07,0,12,0
6,96252,爱情万岁,2000-07-07,1,12,0
8,96291,人生海海,2001-07-07,2,12,0
10,96353,时光机,2003-11-07,3,15,0
5,96368,神的孩子都在跳舞,2004-11-01,4,14,1
3,96397,知足 最真杰作选,2005-08-26,5,11,1
9,15702,为爱而生,2006-12-29,6,13,2
1,36459,后青春期的诗,2008-10-23,7,12,0
4,90142,第二人生,2011-12-16,8,16,2
2,451706,步步 自选作品辑,2013-12-30,9,7,0


In [36]:
album_dict = df_album.to_dict(orient='records')
with open(file_path_prefix + 'album_data.json', 'w', encoding='utf-8') as f:
    json.dump(album_dict, f, ensure_ascii=False, indent=4)


# 页面统计数据

In [39]:
def web_metric(df):
    album_num = df['album_id'].nunique()
    songs_num = df['songs_num'].sum()
    instrumental_num = df['instrumental_num'].sum()
    has_lyric_num = songs_num - instrumental_num

    res = {
        'album_num': int(album_num),
        'songs_num': int(songs_num),
        'instrumental_num': int(instrumental_num),
        'has_lyric_num': int(has_lyric_num)
    }
    return res
web_metric_data = web_metric(df_album)
with open(file_path_prefix + 'web_metric_data.json', 'w', encoding='utf-8') as f:
    json.dump(web_metric_data, f, ensure_ascii=False, indent=4)

# 旧版

In [ ]:
def clear_lyric(df):
    # 曲目，专辑添加唯一id
    df_lyric = df.copy()
    df_lyric['song_id_unique'] = 'song' + df_lyric['song_id'].astype(str)
    df_album_fixed = df_lyric[['album_id', 'album_fixed']].drop_duplicates(subset=['album_fixed'], keep='first').reset_index(drop=True)
    df_album_fixed['album_id_unique'] = 'album' + df_album_fixed['album_id'].astype(int).astype(str)
    df_album_fixed = df_album_fixed[['album_id_unique', 'album_fixed']]
    df_lyric= df_lyric.merge(df_album_fixed, on='album_fixed', how='left')
    # 词唯一id
    df_words = df_lyric[['word', 'pos']].drop_duplicates().reset_index(drop=False)
    df_words['word_id'] = 'word' + df_words['index'].astype(str)
    df_words = df_words.drop('index', axis=1).reset_index(drop=True)
    df_lyric = df_lyric.merge(df_words, on=['word', 'pos'], how='left')
    return df_lyric

df_lyric = clear_lyric(df_lyric_raw)
df_lyric

## 专辑数据

In [ ]:
df_album = df_raw[[
    'album_id', 'album_name', 'album_type', 'release_date', 'album_fixed'
]].copy()
df_album['release_date'] = df_album['release_date'].astype(str)
df_album = df_album.drop_duplicates().reset_index()
df_album

In [ ]:
# 歌曲计数
songs_num = df_raw['album_name'].value_counts()
songs_num

In [ ]:
df_raw['album_name'].unique()

In [ ]:
# 纯音乐数量
instrumental_num = df_raw[df_raw['has_lyric'] ==
                          0]['album_name'].value_counts()
instrumental_num

In [ ]:
# 新歌数量
new_songs_num = df_raw[df_raw['is_duplicate'] ==
                       0]['album_name'].value_counts()
new_songs_num

In [ ]:
df_album = df_album.merge(songs_num, on='album_name', how='left').rename(columns={'count': 'songs_num'})
df_album

In [ ]:
df_album = df_album.merge(
    instrumental_num, on='album_name',
    how='left').rename(columns={'count': 'instrumental_num'}).fillna(0)
df_album 

In [ ]:
df_album = df_album.merge(
    new_songs_num, on='album_name',
    how='left').rename(columns={'count': 'new_songs_num'}).fillna(0)
df_album 

In [ ]:
album_dict = df_album.to_dict(orient='records')
with open('output/album_data.json', 'w', encoding='utf-8') as f:
    json.dump(album_dict, f, ensure_ascii=False, indent=4)


In [ ]:
df_album['album_fixed'].tolist()

## 曲目数据

In [ ]:
df_songs = df_raw[df_raw['is_duplicate'] == 0][[
    'album_fixed', 'song_name'
]].drop_duplicates().reset_index(drop=True).copy()
df_songs

In [ ]:
songs_dict = df_songs.to_dict(orient='records')
with open('output/songs_data.json', 'w', encoding='utf-8') as f:
    json.dump(songs_dict, f, ensure_ascii=False, indent=4)

In [ ]:
df_lyric_raw

# 词图

## 二分图布局

In [ ]:
# 120度弧线布局
def generate_symmetric_bipartite_layout(nodes):
    coords = {}
    
    # --- 1. 参数定义 ---
    # 定义两条对称弧线的几何参数
    arc_radius = 800
    arc_span = math.pi / 1.5  # 约 120 度
    # 计算弧线的垂直跨度 (用于决定 Word 长度)
    arc_vertical_span = 2 * arc_radius * math.sin(arc_span / 2)
    
    # --- 2. Word 节点 (画布中央直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧线跨度的 90%
    target_word_height = arc_vertical_span * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心向两端扩散逻辑: 0->0, 1->1, 2->-1, 3->2...
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        
        # Word 位于 x=0，且在 y 轴居中
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (两侧对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    # 将 Song 平均分为两组：左侧弧和右侧弧
    song_columns = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 弧线布局配置：[左侧弧, 右侧弧]
    # 左侧弧圆心在正 X，向左弯曲；右侧弧圆心在负 X，向右弯曲
    configs = [
        {"center_x": 0, "direction": -1}, # 右侧弧 (位于 Word 右侧)
        {"center_x": 0, "direction": 1}  # 左侧弧 (位于 Word 左侧)
    ]

    for col_idx, col_items in enumerate(song_columns):
        config = configs[col_idx]
        num_in_col = len(col_items)
        if num_in_col == 0: continue
        
        for row_idx, node in enumerate(col_items):
            # 从上到下均匀分布角度
            if num_in_col > 1:
                angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span
            else:
                angle = 0
            
            # 计算坐标
            # cos(angle) 决定 X 偏移，direction 决定是在圆心左侧还是右侧
            x = config["center_x"] + config["direction"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


In [ ]:
# 112度弧形布局
def generate_embracing_layout(nodes):
    coords = {}
    
    # --- 1. 几何参数设定 ---
    arc_radius = 800           # 半径
    arc_span = math.pi / 1.6   # 弧度张角 (约112度)
    # 弧开口端点距离中心直线的水平间距
    horizontal_gap = 300       
    
    # 计算弧线端点的 Y 轴跨度 (用于对齐 Word)
    # y = r * sin(theta)
    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height
    
    # --- 2. Word 节点 (居中直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧垂直跨度的 90%
    target_word_height = arc_total_height * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心扩散排序
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (开口向内的对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 配置说明：
    # 为了让开口面向直线 (x=0)：
    # 左侧弧的圆心要在右侧，x 坐标为 (horizontal_gap + radius * cos(half_span))
    # 右侧弧的圆心要在左侧，x 坐标为 -(horizontal_gap + radius * cos(half_span))
    
    # 计算圆心位置，使得弧的端点正好落在 horizontal_gap 线上
    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1, "name": "Right"}, # 右侧弧，圆心在左，向右弯
        {"c_x": 0, "dir": -1, "name": "Left"}   # 左侧弧，圆心在右，向左弯
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)
        
        for row_idx, node in enumerate(col_items):
            # 角度分布
            angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span if num_in_col > 1 else 0
            
            # 计算 X: 圆心 + 方向 * (半径 * cos(角度))
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


## 数据处理

In [ ]:
# 歌曲表
df_songs = df_lyric[['song_id', 'song_name', 'album_id', 'album_fixed', 'song_id_unique', 'album_id_unique']].drop_duplicates(subset=['song_id', 'album_fixed'], keep='first').reset_index(drop=True)
df_songs

In [ ]:
# 特定词性的数据集
def get_subset_data(df_lyric_raw, pos, words_num, is_starts_with):
    df_lyric = clear_lyric(df_lyric_raw)
    words_set = word_count_by_pos(df_lyric_raw, pos=pos, words_num=words_num, is_starts_with=is_starts_with)
    df_lyric_subset = df_lyric[df_lyric['pos'].str.startswith(pos, na=False)] if is_starts_with else df_lyric[df_lyric['pos'] == pos]
    df_lyric_subset = df_lyric_subset[df_lyric_subset['word'].isin(words_set['word'].tolist())].reset_index(drop=True)
    return df_lyric_subset

## json数据输出

In [ ]:
def get_word_subset_graph(G_words_subset, df_lyric_subset):    
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'song',
        }
        nodes_subset.append(n_dict)
    pos = generate_embracing_layout(nodes_subset)
    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_lyric_subset[df_lyric_subset['word_id'] == node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] ={ 'cluster': '词' } 
        elif 'song' in node:
            nodes_dict['label'] = df_lyric_subset[df_lyric_subset['song_id_unique'] == node]['song_name'].values[0]
            nodes_dict['node_type'] = 'song'
            nodes_dict['album_id'] = df_lyric_subset[df_lyric_subset['song_id_unique'] == node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_lyric_subset[df_lyric_subset['song_id_unique'] == node]['album_fixed'].values[0]
            nodes_dict['data'] ={ 'cluster': df_lyric_subset[df_lyric_subset['song_id_unique'] == node]['album_fixed'].values[0] } 
        word_graph_dict['nodes'].append(nodes_dict)
    for edeg in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edeg[0]
        edges_dict['target'] = edeg[1]
        for i in edeg:
            if 'song' in i:
                edges_dict['album'] = df_lyric_subset[df_lyric_subset['song_id_unique'] == i]['album_fixed'].values[0]
            else:
                edges_dict['album'] = ""
            
        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

In [ ]:
df_lyric = clear_lyric(df_lyric_raw)

In [ ]:
df_lyric_subset = get_subset_data(df_lyric_raw, 'a', words_num=50, is_starts_with=True)
df_lyric_subset

In [ ]:
G_words = nx.from_pandas_edgelist(df_lyric, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_words_subset = nx.from_pandas_edgelist(df_lyric_subset, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_songs = nx.from_pandas_edgelist(df_lyric, 'song_id_unique', 'album_id_unique', edge_attr=True, create_using=nx.Graph())
G_words.number_of_nodes(), G_words_subset.number_of_nodes(), G_songs.number_of_nodes()

In [ ]:
word_graph_dict = get_word_subset_graph(G_words_subset, df_lyric_subset)

In [ ]:
with open('output/word_graph_data.json', 'w', encoding='utf-8') as f:
    json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)

In [ ]:
df_lyric_subset

二分图布局，输入节点id, 类型，节点度，输出坐标
* word节点，在画布左侧排2列直线，按照度从中心到边缘排
* song节点，在画布右侧排3列弧线，按照顺序从上到下排，需要指定每一列的节点数据

In [ ]:
df_song_subset = df_lyric_subset[['song_id_unique', 'album_id_unique']].drop_duplicates()
df_song_subset

In [ ]:
album_list = df_lyric_subset['album_id_unique'].unique()
album_list

In [ ]:
df_song_subset[df_song_subset['album_id_unique'] == 'album38315']['song_id_unique']

In [ ]:
album_group = [['album38315', 'album38308'], 
               ['album38297', 'album38276', 'album38259', 'album38241'], 
       ['album38235', 'album2040001',
       'album34746073', 'album38247', 'album2740205']]
song_cols = [[], [], []]
for i, v in enumerate(album_group):
    for vi in v:
        song_cols[i].extend(df_song_subset[df_song_subset['album_id_unique'] == vi]['song_id_unique'].tolist()) 
    # songs = df_song_subset[df_song_subset['album_id_unique'] == i]['song_id_unique'].tolist()
    # 将songs中的元素加入song_cols[0]中
    # song_cols[i].extend(songs)
song_cols


## 绘图

In [ ]:
# 合并G_word和G_songs
G_lyric = nx.compose(G_words, G_songs)
G_lyric.number_of_nodes()

In [ ]:
G_lyric_subset = nx.compose(G_words_subset, G_songs)

In [ ]:
nx.is_connected(G_words_subset)

In [ ]:
# 绘图
def draw_net(G, k=None, pos_type="spring"):
    if pos_type == "spring":
        pos = nx.spring_layout(G, k=k)
    if pos_type == "twopi":
        pos = nx.nx_agraph.graphviz_layout(G, prog="twopi", args="")
    if pos_type == "shell":
        shells= [[], [], []]
        for i in G.nodes():
            if i.startswith("album"):
                shells[0].append(i)
            elif i.startswith("song"):
                shells[1].append(i)
            else:
                shells[2].append(i)
        pos = nx.shell_layout(G, shells)

    plt.figure(figsize=(10, 10))
    nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_size=1,
        edge_color="gainsboro",
        alpha=0.4,
    )
    plt.title("Lyric Network")
    plt.axis('off')
    plt.show()
    print(G.number_of_nodes(), G.number_of_edges())

In [ ]:
draw_net(G_lyric_subset)

In [ ]:
draw_net(G_words_subset, pos_type='shell')

In [ ]:
draw_net(G_lyric_subset)

In [ ]:
draw_net(G_lyric)

In [ ]:
D_words_subset = {
    'nodes': list(G_words_subset.nodes()),
    'edges': list(G_words_subset.edges())}
with open('output/words_subset.json', 'w', encoding='utf-8') as f:
    json.dump(D_words_subset, f, ensure_ascii=False, indent=4)
